# Plan d.vi -- Estimation

Estimates the model(s) selected by step v's decision branch: Model A (static OLS) is always
estimated as the literal-thesis baseline; step v resolved **Branch B** (mixed I(0)/I(1), no I(2)
regressor), so Model B (ARDL(p,q) bounds testing / conditional ECM) is also estimated as the
**primary** model, alongside a first-differenced OLS for comparison.

See `docs/2_plan/analysis/vi_estimation.md` for the full spec.

**Input** (`outputs/`):
- `analysis_frame_1990_2024.csv` (from step i)
- `modeling_path_decision.csv` (from step v -- branch, per-variable I(0)/I(1)/I(2) classification)

**Outputs** (`outputs/`):
- `model_a_static_ols_coefficients.csv`, `model_a_static_ols_fit_stats.csv`
- `ardl_lag_selection.csv`, `ardl_bounds_test.csv`, `ardl_long_run_coefficients.csv`,
  `ardl_error_correction_term.csv`, `ardl_short_run_coefficients.csv` (AIC-selected grid result)
- `ardl_capped_1_1_bounds_test.csv`, `ardl_capped_1_1_long_run_coefficients.csv`,
  `ardl_capped_1_1_error_correction_term.csv`, `ardl_capped_1_1_short_run_coefficients.csv`
  (fixed ARDL(1,1), no grid search -- maximum-df robustness check, see Step B8)
- `model_b_first_differenced_ols_coefficients.csv`, `model_b_first_differenced_ols_fit_stats.csv`
- `model_comparison_side_by_side.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tools import add_constant
from statsmodels.tsa.ardl import ARDL, UECM

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

FRAME_IN = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
DECISION_IN = OUTPUT_DIR / "modeling_path_decision.csv"

MODEL_A_COEF_OUT = OUTPUT_DIR / "model_a_static_ols_coefficients.csv"
MODEL_A_FIT_OUT = OUTPUT_DIR / "model_a_static_ols_fit_stats.csv"
LAG_SELECTION_OUT = OUTPUT_DIR / "ardl_lag_selection.csv"
BOUNDS_TEST_OUT = OUTPUT_DIR / "ardl_bounds_test.csv"
LONG_RUN_OUT = OUTPUT_DIR / "ardl_long_run_coefficients.csv"
ECT_OUT = OUTPUT_DIR / "ardl_error_correction_term.csv"
SHORT_RUN_OUT = OUTPUT_DIR / "ardl_short_run_coefficients.csv"
CAPPED_BOUNDS_TEST_OUT = OUTPUT_DIR / "ardl_capped_1_1_bounds_test.csv"
CAPPED_LONG_RUN_OUT = OUTPUT_DIR / "ardl_capped_1_1_long_run_coefficients.csv"
CAPPED_ECT_OUT = OUTPUT_DIR / "ardl_capped_1_1_error_correction_term.csv"
CAPPED_SHORT_RUN_OUT = OUTPUT_DIR / "ardl_capped_1_1_short_run_coefficients.csv"
DIFF_OLS_COEF_OUT = OUTPUT_DIR / "model_b_first_differenced_ols_coefficients.csv"
DIFF_OLS_FIT_OUT = OUTPUT_DIR / "model_b_first_differenced_ols_fit_stats.csv"
COMPARISON_OUT = OUTPUT_DIR / "model_comparison_side_by_side.csv"

SIGNIFICANCE_LEVELS = [(0.01, "***"), (0.05, "**"), (0.10, "*")]


def stars(p_value: float) -> str:
    for threshold, mark in SIGNIFICANCE_LEVELS:
        if p_value < threshold:
            return mark
    return ""


REGRESSORS = {
    "DIVP": "divp",
    "DIVM": "divm",
    "INF": "inflation_rate_pct",
    "EXR": "exchange_rate",
    "log(FDI)": "log_fdi",
    "SHOCK": "shock",
}

## Step 1 -- Load inputs and confirm the branch decision from step v

In [2]:
frame = pd.read_csv(FRAME_IN).set_index("year")
assert frame.shape[0] == 35, f"expected 35-row analysis frame, got {frame.shape[0]}"

eri = frame["eri"].rename("ERI")
X = frame[list(REGRESSORS.values())].rename(columns={v: k for k, v in REGRESSORS.items()})

decision = pd.read_csv(DECISION_IN).iloc[0]
branch = decision["branch"]
i1_vars = [v.strip() for v in str(decision["i1_variables"]).split(",") if v.strip()]

print(f"Branch from step v: {branch}")
print(f"models_to_estimate: {decision['models_to_estimate']}")
print(f"I(1) regressors (spurious-regression risk in Model A / meaningful long-run coefficients in Model B): {i1_vars}")

if branch == "C":
    raise RuntimeError(
        "Branch C was selected in step v -- ARDL bounds testing is invalid with an I(2) "
        "regressor and estimation cannot proceed. This notebook stops here per the plan's "
        "'don't guess' instruction; do not silently work around an I(2) result."
    )

Branch from step v: B
models_to_estimate: ARDL(p,q) bounds testing (primary model); static OLS on levels (literal-thesis comparison); OLS on first-differenced variables (comparison).
I(1) regressors (spurious-regression risk in Model A / meaningful long-run coefficients in Model B): ['DIVP', 'DIVM', 'EXR', 'log(FDI)']


## Model A -- Static OLS (always estimated, regardless of branch)

```
ERI = β0 + β1·DIVP + β2·DIVM + β3·INF + β4·EXR + β5·log(FDI) + β6·SHOCK + ε
```

Estimated via `statsmodels.api.OLS` on the full 1990-2024 sample (35 obs), with a constant --
this is the exact package/function the user cross-checks against EViews' `LS` output. This
model is reported regardless of the branch outcome, as the "what a naive reader would conclude"
comparison point against Model B.

In [3]:
design_a = add_constant(X, has_constant="add")
model_a = sm.OLS(eri, design_a).fit()

model_a_coefs = pd.DataFrame({
    "term": model_a.params.index,
    "coef": model_a.params.values,
    "std_err": model_a.bse.values,
    "t_stat": model_a.tvalues.values,
    "p_value": model_a.pvalues.values,
})
model_a_coefs["significance"] = model_a_coefs["p_value"].map(stars)
model_a_coefs.to_csv(MODEL_A_COEF_OUT, index=False)
print(f"Written -> {MODEL_A_COEF_OUT}")
model_a_coefs

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_a_static_ols_coefficients.csv


,term,coef,std_err,t_stat,p_value,significance
0,const,-2.334482,0.730522,-3.195634,0.003443,***
1,DIVP,1.268049,0.602285,2.105398,0.044354,**
2,DIVM,-0.805481,0.887658,-0.907423,0.371927,
3,INF,-0.004062,0.002733,-1.485997,0.148454,
4,EXR,-0.000941,0.000408,-2.307866,0.028610,**
5,log(FDI),0.142768,0.031814,4.487613,0.000112,***
6,SHOCK,-0.020333,0.068522,-0.296735,0.768859,


In [4]:
model_a_fit_stats = pd.DataFrame([{
    "package_function": "statsmodels.api.OLS",
    "n_obs": int(model_a.nobs),
    "r_squared": model_a.rsquared,
    "adj_r_squared": model_a.rsquared_adj,
    "f_statistic": model_a.fvalue,
    "f_pvalue": model_a.f_pvalue,
    "aic": model_a.aic,
    "bic": model_a.bic,
}])
model_a_fit_stats.to_csv(MODEL_A_FIT_OUT, index=False)
print(f"Written -> {MODEL_A_FIT_OUT}")
model_a_fit_stats.T

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_a_static_ols_fit_stats.csv


,0
package_function,statsmodels.api.OLS
n_obs,35
r_squared,0.598389
adj_r_squared,0.51233
f_statistic,6.953207
f_pvalue,0.000133
aic,-43.376734
bic,-32.489297


## Model B -- ARDL(p,q) bounds testing (Branch B: primary model)

Conditional Error-Correction (ECM) form:

```
ΔERI_t = α0 + Σγᵢ·ΔERI_(t-i) + Σδⱼ·ΔX_(t-j) + θ1·ERI_(t-1) + θ2·DIVP_(t-1) + θ3·DIVM_(t-1)
         + θ4·INF_(t-1) + θ5·EXR_(t-1) + θ6·log(FDI)_(t-1) + θ7·SHOCK_(t-1) + ε_t
```

Estimated via `statsmodels.tsa.ardl.UECM` (the unrestricted-ECM reparametrization of ARDL(p,q),
with the built-in Pesaran/Shin/Smith bounds test) -- the direct equivalent of EViews' ARDL
wizard in "conditional ECM" output form, which is what the user cross-checks against.

### Step B1 -- Lag selection: grid search over p (own lags of ERI) and q (lags of each regressor)

Per the plan, the maximum lag is capped at 1-2 given the ~34-35 observation sample. `q` is
applied uniformly across all six regressors (rather than optimized independently per regressor)
-- with only ~34 observations, letting each of the six regressors pick its own lag independently
would explode the search space (searched by `statsmodels.tsa.ardl.ardl_select_order` over
thousands of combinations) and risks overfitting the lag structure to noise. `q = 0` is excluded
from the grid: `UECM` requires at least one lag for every exogenous regressor to construct the
conditional-ECM short-run terms (`ValueError: All included exog variables must have a lag length
>= 1`), so `q = 0` is not a candidate once the bounds-test parametrization is the target, not a
judgment call.

AIC and BIC are compared on a common sample (`hold_back` fixed at the largest candidate `p`, so
all candidates are fit on the same 33 observations -- otherwise AIC/BIC are not comparable across
different lag lengths).

In [5]:
MAX_P = 2
lag_grid = []
for p in (1, 2):
    for q in (1, 2):
        fit = ARDL(eri, lags=p, exog=X, order=q, trend="c", hold_back=MAX_P).fit()
        lag_grid.append({
            "p": p, "q": q,
            "n_obs": int(fit.nobs),
            "k_params": len(fit.params),
            "df_resid": int(fit.df_resid),
            "aic": fit.aic,
            "bic": fit.bic,
        })

lag_selection = pd.DataFrame(lag_grid)
lag_selection.to_csv(LAG_SELECTION_OUT, index=False)
print(f"Written -> {LAG_SELECTION_OUT}")
lag_selection

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_lag_selection.csv


,p,q,n_obs,k_params,df_resid,aic,bic
0,1,1,33,14,19,-32.305969,-9.858356
1,1,2,33,20,13,-40.295026,-8.868367
2,2,1,33,15,18,-35.152112,-11.207991
3,2,2,33,21,12,-39.795651,-6.872484


In [6]:
aic_best = lag_selection.loc[lag_selection["aic"].idxmin()]
bic_best = lag_selection.loc[lag_selection["bic"].idxmin()]

aic_pq = (int(aic_best["p"]), int(aic_best["q"]))
bic_pq = (int(bic_best["p"]), int(bic_best["q"]))

print(f"AIC-optimal (p,q) = {aic_pq}  (AIC={aic_best['aic']:.3f}, {int(aic_best['k_params'])} params, {int(aic_best['df_resid'])} residual df)")
print(f"BIC-optimal (p,q) = {bic_pq}  (BIC={bic_best['bic']:.3f}, {int(bic_best['k_params'])} params, {int(bic_best['df_resid'])} residual df)")

if aic_pq != bic_pq:
    print()
    print(f"FLAG: AIC and BIC disagree on the optimal lag -- AIC picks {aic_pq}, BIC picks {bic_pq}.")
    print(f"Default used per the plan's stated default: AIC -> (p,q) = {aic_pq}.")
    print(f"BIC-implied alternative (p,q) = {bic_pq} is carried forward as a robustness check below (Step B8),")
    print("not silently discarded.")
    print()
    print(
        f"FLAG (degrees of freedom): the AIC-preferred specification uses {int(aic_best['k_params'])} "
        f"parameters against only {int(aic_best['n_obs'])} observations ({int(aic_best['df_resid'])} residual "
        "df) -- a thin margin, which is exactly the risk the plan's 1-2 lag cap is meant to guard against. "
        "This is stated explicitly rather than silently accepted."
    )
    chosen_p, chosen_q = aic_pq
else:
    print("AIC and BIC agree -- no tiebreak needed.")
    chosen_p, chosen_q = aic_pq

AIC-optimal (p,q) = (1, 2)  (AIC=-40.295, 20 params, 13 residual df)
BIC-optimal (p,q) = (2, 1)  (BIC=-11.208, 15 params, 18 residual df)

FLAG: AIC and BIC disagree on the optimal lag -- AIC picks (1, 2), BIC picks (2, 1).
Default used per the plan's stated default: AIC -> (p,q) = (1, 2).
BIC-implied alternative (p,q) = (2, 1) is carried forward as a robustness check below (Step B8),
not silently discarded.

FLAG (degrees of freedom): the AIC-preferred specification uses 20 parameters against only 33 observations (13 residual df) -- a thin margin, which is exactly the risk the plan's 1-2 lag cap is meant to guard against. This is stated explicitly rather than silently accepted.


### Step B2 -- Estimate the UECM at the AIC-selected (p,q)

Fit on the natural (non-hold_back-restricted) sample so the reported model uses as much of the
short sample as the selected lag structure allows.

In [7]:
uecm = UECM(eri, lags=chosen_p, exog=X, order=chosen_q, trend="c").fit()
print(f"UECM(p={chosen_p}, q={chosen_q}) -- n_obs={int(uecm.nobs)}, df_resid={int(uecm.df_resid)}, "
      f"aic={uecm.aic:.3f}, bic={uecm.bic:.3f}")

full_params = pd.DataFrame({
    "term": uecm.params.index,
    "coef": uecm.params.values,
    "std_err": uecm.bse.values,
    "t_stat": uecm.tvalues.values,
    "p_value": uecm.pvalues.values,
})
full_params["significance"] = full_params["p_value"].map(stars)
full_params

UECM(p=1, q=2) -- n_obs=34, df_resid=14, aic=-43.804, bic=-11.750


,term,coef,std_err,t_stat,p_value,significance
0,const,-2.030847,1.671609,-1.214906,0.244495,
1,ERI.L1,-1.005993,0.270580,-3.717918,0.002294,***
2,DIVP.L1,-0.019534,1.032052,-0.018927,0.985166,
3,DIVM.L1,-1.248701,2.014033,-0.620000,0.545218,
4,INF.L1,0.007438,0.009705,0.766392,0.456172,
5,EXR.L1,-0.000307,0.001305,-0.235102,0.817534,
6,log(FDI).L1,0.188030,0.101945,1.844419,0.086384,*
7,SHOCK.L1,0.033205,0.183272,0.181180,0.858823,
8,D.DIVP.L0,0.040518,1.715958,0.023612,0.981495,
9,D.DIVP.L1,2.533764,1.601102,1.582512,0.135855,


### Step B3 -- Bounds F-test

Joint Wald test of `H0: θ1 = θ2 = ... = θ7 = 0` (the lagged-level terms), via
`UECMResults.bounds_test`, against Pesaran, Shin & Smith (2001) critical value bounds for k = 6
regressors (`DIVP, DIVM, INF, EXR, log(FDI), SHOCK` -- the six X's; `ERI` itself is the
dependent variable's own lagged level, not counted in k).

**Case used: Case 3 -- "constant included in the model but not in the test" (unrestricted
intercept, no trend).** This matches the model spec exactly: `trend="c"` includes a constant in
the UECM, and there is no trend term at all, so the intercept is left unrestricted (not forced
into the cointegrating relationship) and no trend case (4/5) applies. Stated explicitly since
picking the wrong case is a common EViews-vs-Python mismatch source.

In [8]:
BOUNDS_CASE = 3
bounds_result = uecm.bounds_test(case=BOUNDS_CASE)

bounds_table = bounds_result.crit_vals.copy()
bounds_table.columns = ["crit_lower", "crit_upper"]
bounds_table["f_stat"] = bounds_result.stat
bounds_table["between_bounds"] = (bounds_table["f_stat"] > bounds_table["crit_lower"]) & (
    bounds_table["f_stat"] < bounds_table["crit_upper"]
)
bounds_table["above_upper"] = bounds_table["f_stat"] > bounds_table["crit_upper"]
bounds_table["below_lower"] = bounds_table["f_stat"] < bounds_table["crit_lower"]
bounds_table = bounds_table.reset_index().rename(columns={"percentile": "confidence_level_pct"})
bounds_table.insert(0, "case", BOUNDS_CASE)
bounds_table.insert(1, "k_regressors", 6)
bounds_table.to_csv(BOUNDS_TEST_OUT, index=False)
print(f"Written -> {BOUNDS_TEST_OUT}")
bounds_table

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_bounds_test.csv


,case,k_regressors,confidence_level_pct,crit_lower,crit_upper,f_stat,between_bounds,above_upper,below_lower
0,3,6,90.0,2.030543,3.136325,2.822996,True,False,False
1,3,6,95.0,2.328170,3.499913,2.822996,True,False,False
2,3,6,99.0,2.956935,4.251989,2.822996,False,False,True
3,3,6,99.9,3.778834,5.207530,2.822996,False,False,True


In [9]:
row_5pct = bounds_table.loc[bounds_table["confidence_level_pct"] == 95.0].iloc[0]
row_1pct = bounds_table.loc[bounds_table["confidence_level_pct"] == 99.0].iloc[0]

print(f"F-stat = {bounds_result.stat:.4f}")
print(f"At 5%: lower={row_5pct.crit_lower:.3f}, upper={row_5pct.crit_upper:.3f}")
print(f"At 1%: lower={row_1pct.crit_lower:.3f}, upper={row_1pct.crit_upper:.3f}")
print()

if row_5pct["above_upper"]:
    cointegration_status = "confirmed"
    print("F-stat is ABOVE the upper bound at 5% -> cointegration CONFIRMED. Long-run coefficients are interpretable.")
elif row_5pct["below_lower"]:
    cointegration_status = "not confirmed"
    print("F-stat is BELOW the lower bound at 5% -> NO cointegration. Falling back to the short-run/differenced-OLS interpretation; long-run coefficients are not reported as confirmed.")
else:
    cointegration_status = "inconclusive"
    print("F-stat falls BETWEEN the bounds at 5% -> INCONCLUSIVE. Per the plan, this is reported as such and not forced toward either conclusion.")
    if row_1pct["below_lower"]:
        print("Note: at the stricter 1% level the stat falls BELOW the lower bound (no cointegration at 1%),")
        print("which leans the inconclusive 5%-level result toward caution rather than toward confirmation.")

print(f"\ncointegration_status = '{cointegration_status}'")

F-stat = 2.8230
At 5%: lower=2.328, upper=3.500
At 1%: lower=2.957, upper=4.252

F-stat falls BETWEEN the bounds at 5% -> INCONCLUSIVE. Per the plan, this is reported as such and not forced toward either conclusion.
Note: at the stricter 1% level the stat falls BELOW the lower bound (no cointegration at 1%),
which leans the inconclusive 5%-level result toward caution rather than toward confirmation.

cointegration_status = 'inconclusive'


### Step B4 -- Long-run coefficients

Derived as `βₖ_LR = −θₖ/θ1` via `UECMResults.ci_params` (statsmodels' built-in normalized
cointegrating-relationship parametrization). Standard errors via **the delta method**
(`UECMResults.ci_bse`, which applies the analytic delta-method Jacobian to the coefficient
covariance matrix -- not bootstrap), matching the plan's requirement to state which method is
used.

**Caveat, stated explicitly per the plan's "do not force a conclusion" instruction:** the bounds
test above is inconclusive at 5% (and leans toward no-cointegration at 1%), so cointegration is
**not confirmed**. These long-run coefficients are reported below for transparency and for the
Chapter 4 side-by-side comparison, but they should **not** be interpreted as confirmed long-run
effects -- only as what the ARDL long-run parametrization implies conditional on cointegration
holding, which the data do not clearly establish.

In [10]:
long_run = pd.DataFrame({
    "term": uecm.ci_params.index,
    "long_run_coef": uecm.ci_params.values,
    "std_err_delta_method": uecm.ci_bse.values,
    "t_stat": uecm.ci_tvalues.values,
    "p_value": uecm.ci_pvalues.values,
})
long_run = long_run[long_run["term"] != "ERI"].reset_index(drop=True)  # ERI is normalized to 1 by construction
long_run["significance"] = long_run["p_value"].map(stars)
long_run["cointegration_confirmed_by_bounds_test"] = cointegration_status == "confirmed"
long_run.to_csv(LONG_RUN_OUT, index=False)
print(f"Written -> {LONG_RUN_OUT}")
long_run

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_long_run_coefficients.csv


,term,long_run_coef,std_err_delta_method,t_stat,p_value,significance,cointegration_confirmed_by_bounds_test
0,const,2.018749,1.336996,1.509914,0.131065,,False
1,DIVP,0.019417,1.027557,0.018897,0.984924,,False
2,DIVM,1.241263,2.109532,0.588407,0.556259,,False
3,INF,-0.007394,0.008895,-0.831191,0.405865,,False
4,EXR,0.000305,0.001277,0.238727,0.811318,,False
5,log(FDI),-0.186910,0.095197,-1.963407,0.049599,**,False
6,SHOCK,-0.033007,0.186123,-0.177342,0.859240,,False


### Step B5 -- Error-correction term

`θ1`, the `ERI.L1` coefficient -- should be negative and significant, ideally between −1 and 0.
Its magnitude is the speed of adjustment back to the long-run resilience level after a shock
(the "restorative capacity" framing in Ch. 2.3).

In [11]:
ect_coef = uecm.params["ERI.L1"]
ect_se = uecm.bse["ERI.L1"]
ect_t = uecm.tvalues["ERI.L1"]
ect_p = uecm.pvalues["ERI.L1"]
ect_significant_negative = (ect_coef < 0) and (ect_p < 0.05)
ect_in_expected_range = -1.0 < ect_coef < 0.0

ect_table = pd.DataFrame([{
    "term": "ERI.L1 (error-correction term, theta1)",
    "coef": ect_coef,
    "std_err": ect_se,
    "t_stat": ect_t,
    "p_value": ect_p,
    "significance": stars(ect_p),
    "negative_and_significant_at_5pct": ect_significant_negative,
    "within_minus1_to_0": ect_in_expected_range,
}])
ect_table.to_csv(ECT_OUT, index=False)
print(f"Written -> {ECT_OUT}")

print(f"theta1 (ERI.L1) = {ect_coef:.4f}, se={ect_se:.4f}, t={ect_t:.4f}, p={ect_p:.4f}")
if not ect_significant_negative:
    print("FLAG: theta1 is not negative-and-significant at 5% -- this would undermine the cointegration story.")
elif not ect_in_expected_range:
    print(f"FLAG: theta1 = {ect_coef:.4f} is negative and significant but falls outside the expected (-1, 0) "
          "range -- adjustment overshoots the long-run level each period rather than converging monotonically. "
          "Stated explicitly rather than silently reported as unremarkable.")
else:
    print("theta1 is negative, significant at 5%, and within the expected (-1, 0) range.")

if ect_significant_negative and cointegration_status != "confirmed":
    print()
    print("FLAG (internal tension): the error-correction term looks well-behaved (negative, significant) even "
          "though the bounds test did NOT confirm cointegration at 5%. This is the converse of the scenario the "
          "plan calls out (a confirmed bounds test with a broken ECT) but the same principle applies -- it is "
          "flagged here as worth discussing rather than treated as full corroboration of cointegration on its own; "
          "a significant ECT alone is not sufficient evidence of cointegration without bounds-test confirmation.")

ect_table

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_error_correction_term.csv
theta1 (ERI.L1) = -1.0060, se=0.2706, t=-3.7179, p=0.0023
FLAG: theta1 = -1.0060 is negative and significant but falls outside the expected (-1, 0) range -- adjustment overshoots the long-run level each period rather than converging monotonically. Stated explicitly rather than silently reported as unremarkable.

FLAG (internal tension): the error-correction term looks well-behaved (negative, significant) even though the bounds test did NOT confirm cointegration at 5%. This is the converse of the scenario the plan calls out (a confirmed bounds test with a broken ECT) but the same principle applies -- it is flagged here as worth discussing rather than treated as full corroboration of cointegration on its own; a significant ECT alone is not sufficient evidence of cointegration without bounds-test confirmation.


,term,coef,std_err,t_stat,p_value,significance,negative_and_significant_at_5pct,within_minus1_to_0
0,"ERI.L1 (error-correction term, theta1)",-1.005993,0.27058,-3.717918,0.002294,***,True,False


### Step B6 -- Short-run coefficients

All `Δ`-term coefficients (`γᵢ` on `ΔERI` lags, `δⱼ` on `ΔX` lags), reported as supplementary
evidence on short-run dynamics and adjustment speed.

In [12]:
short_run_terms = [t for t in uecm.params.index if t.startswith("D.")]
short_run = pd.DataFrame({
    "term": short_run_terms,
    "coef": uecm.params[short_run_terms].values,
    "std_err": uecm.bse[short_run_terms].values,
    "t_stat": uecm.tvalues[short_run_terms].values,
    "p_value": uecm.pvalues[short_run_terms].values,
})
short_run["significance"] = short_run["p_value"].map(stars)
short_run.to_csv(SHORT_RUN_OUT, index=False)
print(f"Written -> {SHORT_RUN_OUT}")
short_run

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_short_run_coefficients.csv


,term,coef,std_err,t_stat,p_value,significance
0,D.DIVP.L0,0.040518,1.715958,0.023612,0.981495,
1,D.DIVP.L1,2.533764,1.601102,1.582512,0.135855,
2,D.DIVM.L0,-1.123673,1.678841,-0.669315,0.514182,
3,D.DIVM.L1,-0.262806,1.008628,-0.260558,0.798227,
4,D.INF.L0,0.008804,0.008504,1.035198,0.318128,
5,D.INF.L1,0.002579,0.006947,0.371311,0.715965,
6,D.EXR.L0,-0.006061,0.003401,-1.781999,0.096443,*
7,D.EXR.L1,-0.004277,0.004802,-0.890721,0.388130,
8,D.log(FDI).L0,0.104157,0.103639,1.004992,0.331956,
9,D.log(FDI).L1,-0.029417,0.075904,-0.387554,0.704175,


### Step B7 -- Robustness: does the AIC/BIC lag disagreement change the substantive conclusion?

Re-runs the bounds test and error-correction term on the BIC-preferred `(p,q)` from Step B1, to
check whether the AIC-vs-BIC disagreement flagged above actually matters for the substantive
story (inconclusive bounds test; negative-significant ECT). This is also revisited more formally
as a robustness check in step viii; this is a quick sanity check in-place.

In [13]:
if bic_pq != aic_pq:
    bic_p, bic_q = bic_pq
    uecm_bic = UECM(eri, lags=bic_p, exog=X, order=bic_q, trend="c").fit()
    bounds_bic = uecm_bic.bounds_test(case=BOUNDS_CASE)
    crit_bic_5pct = bounds_bic.crit_vals.loc[95.0]

    if bounds_bic.stat > crit_bic_5pct["upper"]:
        bic_status = "confirmed"
    elif bounds_bic.stat < crit_bic_5pct["lower"]:
        bic_status = "not confirmed (below lower bound)"
    else:
        bic_status = "inconclusive (between bounds)"

    ect_bic_coef = uecm_bic.params["ERI.L1"]
    ect_bic_p = uecm_bic.pvalues["ERI.L1"]

    print(f"BIC-alternative UECM(p={bic_p}, q={bic_q}): n_obs={int(uecm_bic.nobs)}, df_resid={int(uecm_bic.df_resid)}")
    print(f"Bounds F-stat = {bounds_bic.stat:.4f} (5% bounds: {crit_bic_5pct['lower']:.3f}-{crit_bic_5pct['upper']:.3f}) -> {bic_status}")
    print(f"theta1 (ERI.L1) = {ect_bic_coef:.4f}, p = {ect_bic_p:.4f}")
    print()
    print(
        f"Conclusion: the BIC-alternative specification reaches the SAME substantive conclusion as the "
        f"AIC-selected model -- bounds test {bic_status}, ECT negative and significant. The AIC/BIC lag "
        "disagreement flagged in Step B1 does not change the substantive finding, which is reassuring given "
        "the degrees-of-freedom concern with the AIC-preferred specification."
    )
else:
    print("Not applicable -- AIC and BIC agreed on (p,q) in Step B1.")

BIC-alternative UECM(p=2, q=1): n_obs=33, df_resid=18
Bounds F-stat = 2.9099 (5% bounds: 2.328-3.500) -> inconclusive (between bounds)
theta1 (ERI.L1) = -1.0277, p = 0.0032

Conclusion: the BIC-alternative specification reaches the SAME substantive conclusion as the AIC-selected model -- bounds test inconclusive (between bounds), ECT negative and significant. The AIC/BIC lag disagreement flagged in Step B1 does not change the substantive finding, which is reassuring given the degrees-of-freedom concern with the AIC-preferred specification.


### Step B8 -- Capped ARDL(1,1): fixed lag length, no grid search

The AIC-selected (1,2) specification above uses 20 parameters against 33-34 observations,
leaving only 13-14 residual degrees of freedom -- too thin to be confident the inconclusive
bounds-test result (Step B3) reflects a genuine absence/presence of cointegration rather than a
small-sample power problem. `p=1, q=1` is the leanest lag structure that still fits the ARDL
conditional-ECM form (both own-lag and every regressor's lag are set to the minimum of 1, no
search over alternatives). Re-running the bounds test and error-correction term at this fixed
spec gives a cleaner read, uncontaminated by the parameter-heavy AIC pick.

In [14]:
CAPPED_P, CAPPED_Q = 1, 1
uecm_capped = UECM(eri, lags=CAPPED_P, exog=X, order=CAPPED_Q, trend="c").fit()
print(f"UECM(p={CAPPED_P}, q={CAPPED_Q}) -- n_obs={int(uecm_capped.nobs)}, df_resid={int(uecm_capped.df_resid)}, "
      f"aic={uecm_capped.aic:.3f}, bic={uecm_capped.bic:.3f}")

df_comparison = pd.concat([
    lag_selection[["p", "q", "n_obs", "k_params", "df_resid", "aic", "bic"]],
    pd.DataFrame([{
        "p": CAPPED_P, "q": CAPPED_Q,
        "n_obs": int(uecm_capped.nobs), "k_params": len(uecm_capped.params),
        "df_resid": int(uecm_capped.df_resid), "aic": uecm_capped.aic, "bic": uecm_capped.bic,
    }]),
], ignore_index=True)
df_comparison["spec"] = df_comparison.apply(
    lambda r: f"({int(r.p)},{int(r.q)})"
    + (" [AIC pick, Step B2]" if (int(r.p), int(r.q)) == aic_pq else "")
    + (" [BIC pick, Step B7]" if (int(r.p), int(r.q)) == bic_pq else "")
    + (" [capped, Step B8 -- no search]" if (int(r.p), int(r.q)) == (CAPPED_P, CAPPED_Q) else ""),
    axis=1,
)
print()
print("Degrees-of-freedom comparison across all (p,q) specs touched in this notebook:")
df_comparison[["spec", "n_obs", "k_params", "df_resid", "aic", "bic"]]

UECM(p=1, q=1) -- n_obs=34, df_resid=20, aic=-34.729, bic=-11.834

Degrees-of-freedom comparison across all (p,q) specs touched in this notebook:


,spec,n_obs,k_params,df_resid,aic,bic
0,"(1,1) [capped, Step B8 -- no search]",33,14,19,-32.305969,-9.858356
1,"(1,2) [AIC pick, Step B2]",33,20,13,-40.295026,-8.868367
2,"(2,1) [BIC pick, Step B7]",33,15,18,-35.152112,-11.207991
3,"(2,2)",33,21,12,-39.795651,-6.872484
4,"(1,1) [capped, Step B8 -- no search]",34,14,20,-34.729206,-11.833799


Bounds F-test on the capped ARDL(1,1), same Case 3 (unrestricted intercept, no trend, k=6) as
every other spec in this notebook, checked at all three conventional levels (1%, 5%, 10%).

In [15]:
bounds_capped_result = uecm_capped.bounds_test(case=BOUNDS_CASE)

bounds_capped_table = bounds_capped_result.crit_vals.copy()
bounds_capped_table.columns = ["crit_lower", "crit_upper"]
bounds_capped_table["f_stat"] = bounds_capped_result.stat
bounds_capped_table["between_bounds"] = (bounds_capped_table["f_stat"] > bounds_capped_table["crit_lower"]) & (
    bounds_capped_table["f_stat"] < bounds_capped_table["crit_upper"]
)
bounds_capped_table["above_upper"] = bounds_capped_table["f_stat"] > bounds_capped_table["crit_upper"]
bounds_capped_table["below_lower"] = bounds_capped_table["f_stat"] < bounds_capped_table["crit_lower"]
bounds_capped_table = bounds_capped_table.reset_index().rename(columns={"percentile": "confidence_level_pct"})
bounds_capped_table.insert(0, "case", BOUNDS_CASE)
bounds_capped_table.insert(1, "k_regressors", 6)
bounds_capped_table.to_csv(CAPPED_BOUNDS_TEST_OUT, index=False)
print(f"Written -> {CAPPED_BOUNDS_TEST_OUT}")
bounds_capped_table

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_bounds_test.csv


,case,k_regressors,confidence_level_pct,crit_lower,crit_upper,f_stat,between_bounds,above_upper,below_lower
0,3,6,90.0,2.030543,3.136325,2.365753,True,False,False
1,3,6,95.0,2.328170,3.499913,2.365753,True,False,False
2,3,6,99.0,2.956935,4.251989,2.365753,False,False,True
3,3,6,99.9,3.778834,5.207530,2.365753,False,False,True


In [16]:
for level, pct in [("1%", 99.0), ("5%", 95.0), ("10%", 90.0)]:
    row = bounds_capped_table.loc[bounds_capped_table["confidence_level_pct"] == pct].iloc[0]
    verdict = "ABOVE upper (cointegration)" if row.above_upper else (
        "BELOW lower (no cointegration)" if row.below_lower else "BETWEEN bounds (inconclusive)"
    )
    print(f"At {level}: lower={row.crit_lower:.3f}, upper={row.crit_upper:.3f}, stat={row.f_stat:.4f} -> {verdict}")

row_5pct_capped = bounds_capped_table.loc[bounds_capped_table["confidence_level_pct"] == 95.0].iloc[0]
print()

if row_5pct_capped["above_upper"]:
    capped_cointegration_status = "confirmed"
    print("DECISIVE at 5%: F-stat is ABOVE the upper bound -> cointegration CONFIRMED at the leanest spec.")
elif row_5pct_capped["below_lower"]:
    capped_cointegration_status = "not confirmed"
    print("DECISIVE at 5%: F-stat is BELOW the lower bound -> NO cointegration at the leanest spec. "
          "Cointegration is rejected; long-run coefficients below are not to be interpreted as confirmed.")
else:
    capped_cointegration_status = "inconclusive"
    print(
        "STILL INCONCLUSIVE at p=1,q=1, even with the maximum degrees of freedom this ARDL form allows. "
        "This is a finding, not a dead end: with ~34 annual observations and k=6 regressors, the Pesaran "
        "bounds test may simply lack the power to confirm or rule out cointegration here, independent of "
        "which lag length is chosen. Reported honestly in Chapter 4 as a power limitation of the sample "
        "size relative to the number of regressors, rather than forced toward either conclusion."
    )

print(f"\ncapped_cointegration_status = '{capped_cointegration_status}'")

At 1%: lower=2.957, upper=4.252, stat=2.3658 -> BELOW lower (no cointegration)
At 5%: lower=2.328, upper=3.500, stat=2.3658 -> BETWEEN bounds (inconclusive)
At 10%: lower=2.031, upper=3.136, stat=2.3658 -> BETWEEN bounds (inconclusive)

STILL INCONCLUSIVE at p=1,q=1, even with the maximum degrees of freedom this ARDL form allows. This is a finding, not a dead end: with ~34 annual observations and k=6 regressors, the Pesaran bounds test may simply lack the power to confirm or rule out cointegration here, independent of which lag length is chosen. Reported honestly in Chapter 4 as a power limitation of the sample size relative to the number of regressors, rather than forced toward either conclusion.

capped_cointegration_status = 'inconclusive'


Long-run coefficients and the error-correction term at the capped (1,1) spec -- reported
regardless of whether the bounds test above turned out decisive or still inconclusive, same as
Step B4/B5, with the same "not confirmed" caveat applied if the bounds test did not confirm
cointegration.

In [17]:
long_run_capped = pd.DataFrame({
    "term": uecm_capped.ci_params.index,
    "long_run_coef": uecm_capped.ci_params.values,
    "std_err_delta_method": uecm_capped.ci_bse.values,
    "t_stat": uecm_capped.ci_tvalues.values,
    "p_value": uecm_capped.ci_pvalues.values,
})
long_run_capped = long_run_capped[long_run_capped["term"] != "ERI"].reset_index(drop=True)
long_run_capped["significance"] = long_run_capped["p_value"].map(stars)
long_run_capped["cointegration_confirmed_by_bounds_test"] = capped_cointegration_status == "confirmed"
long_run_capped.to_csv(CAPPED_LONG_RUN_OUT, index=False)
print(f"Written -> {CAPPED_LONG_RUN_OUT}")
long_run_capped

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_long_run_coefficients.csv


,term,long_run_coef,std_err_delta_method,t_stat,p_value,significance,cointegration_confirmed_by_bounds_test
0,const,1.677979,1.329219,1.262379,0.206812,,False
1,DIVP,-1.045092,0.982920,-1.063253,0.287667,,False
2,DIVM,1.205365,1.679532,0.717679,0.472955,,False
3,INF,0.002154,0.007343,0.293397,0.769219,,False
4,EXR,0.000055,0.000890,0.061988,0.950573,,False
5,log(FDI),-0.130140,0.059393,-2.191182,0.028439,**,False
6,SHOCK,0.022792,0.130576,0.174546,0.861437,,False


In [18]:
ect_capped_coef = uecm_capped.params["ERI.L1"]
ect_capped_se = uecm_capped.bse["ERI.L1"]
ect_capped_t = uecm_capped.tvalues["ERI.L1"]
ect_capped_p = uecm_capped.pvalues["ERI.L1"]
ect_capped_significant_negative = (ect_capped_coef < 0) and (ect_capped_p < 0.05)
ect_capped_in_expected_range = -1.0 < ect_capped_coef < 0.0

ect_capped_table = pd.DataFrame([{
    "term": "ERI.L1 (error-correction term, theta1)",
    "coef": ect_capped_coef,
    "std_err": ect_capped_se,
    "t_stat": ect_capped_t,
    "p_value": ect_capped_p,
    "significance": stars(ect_capped_p),
    "negative_and_significant_at_5pct": ect_capped_significant_negative,
    "within_minus1_to_0": ect_capped_in_expected_range,
}])
ect_capped_table.to_csv(CAPPED_ECT_OUT, index=False)
print(f"Written -> {CAPPED_ECT_OUT}")

print(f"theta1 (ERI.L1) = {ect_capped_coef:.4f}, se={ect_capped_se:.4f}, t={ect_capped_t:.4f}, p={ect_capped_p:.4f}")
if not ect_capped_significant_negative:
    print("FLAG: theta1 is not negative-and-significant at 5% at the capped spec.")
elif not ect_capped_in_expected_range:
    print(f"FLAG: theta1 = {ect_capped_coef:.4f} is negative and significant but outside the expected (-1, 0) range.")
else:
    print("theta1 is negative, significant at 5%, and within the expected (-1, 0) range -- "
          "better-behaved than the AIC-selected (1,2) spec's theta1, which fell just outside (-1,0).")

ect_capped_table

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_error_correction_term.csv
theta1 (ERI.L1) = -0.7585, se=0.2378, t=-3.1902, p=0.0046
theta1 is negative, significant at 5%, and within the expected (-1, 0) range -- better-behaved than the AIC-selected (1,2) spec's theta1, which fell just outside (-1,0).


,term,coef,std_err,t_stat,p_value,significance,negative_and_significant_at_5pct,within_minus1_to_0
0,"ERI.L1 (error-correction term, theta1)",-0.758516,0.237764,-3.1902,0.004598,***,True,True


In [19]:
short_run_capped_terms = [t for t in uecm_capped.params.index if t.startswith("D.")]
short_run_capped = pd.DataFrame({
    "term": short_run_capped_terms,
    "coef": uecm_capped.params[short_run_capped_terms].values,
    "std_err": uecm_capped.bse[short_run_capped_terms].values,
    "t_stat": uecm_capped.tvalues[short_run_capped_terms].values,
    "p_value": uecm_capped.pvalues[short_run_capped_terms].values,
})
short_run_capped["significance"] = short_run_capped["p_value"].map(stars)
short_run_capped.to_csv(CAPPED_SHORT_RUN_OUT, index=False)
print(f"Written -> {CAPPED_SHORT_RUN_OUT}")
short_run_capped

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_short_run_coefficients.csv


,term,coef,std_err,t_stat,p_value,significance
0,D.DIVP.L0,1.374798,1.610284,0.853761,0.403348,
1,D.DIVM.L0,-0.948627,1.169724,-0.810983,0.426921,
2,D.INF.L0,0.003660,0.006638,0.551403,0.587466,
3,D.EXR.L0,-0.004547,0.002730,-1.665354,0.111430,
4,D.log(FDI).L0,0.115607,0.058387,1.980016,0.061628,*
5,D.SHOCK.L0,-0.012942,0.086264,-0.150025,0.882247,


### Which ARDL specification is carried forward as "the" ARDL result for step vii diagnostics?

**The capped ARDL(1,1) from Step B8 is carried forward to step vii** (post-estimation
diagnostics), not the AIC-selected grid-search (1,2) spec from Step B2. Reasons:

1. **Degrees of freedom.** Step vii's diagnostics (Breusch-Godfrey, Breusch-Pagan, Ramsey RESET,
   CUSUM/CUSUMSQ) rely on asymptotic distributions that need residual degrees of freedom to be
   trustworthy. The (1,2) spec's 13-14 residual df is too thin for those tests to mean much;
   the (1,1) spec's ~20 residual df is a meaningfully better base to diagnose from.
2. **Same substantive conclusion, more cleanly reached.** All three specs touched in this
   notebook -- AIC's (1,2), BIC's (2,1), and the capped (1,1) -- agree: bounds test inconclusive
   at 5% (leaning toward no-cointegration at 1%), error-correction term negative and
   significant. The capped spec reaches this with the least machinery and the least
   overfitting risk, so it is the more defensible "primary" result to hang diagnostics on.
3. **Nothing is discarded.** The grid-searched (1,2) result (Step B2-B6) and the BIC-alternative
   (2,1) result (Step B7) remain saved to their own output files and in the comparison table
   below -- kept for the Chapter 4 write-up to show the robustness check was actually performed,
   not overwritten by this choice.

`outputs/ardl_capped_1_1_*.csv` are the files step vii should read from; `outputs/ardl_*.csv`
(without the `capped_1_1` suffix) remain the AIC-selected grid-search result, retained for the
robustness discussion in step viii.

## Also estimated for comparison -- OLS on first-differenced variables

```
ΔERI = β0 + β1·ΔDIVP + β2·ΔDIVM + β3·ΔINF + β4·ΔEXR + β5·Δlog(FDI) + β6·ΔSHOCK + ε
```

Shows what changes once the trending-regressor spurious-regression risk (from the I(1)
variables `DIVP, DIVM, EXR, log(FDI)`) is removed by differencing, without imposing the full
ARDL/ECM structure. Reported alongside Model A and Model B in the Chapter 4 comparison table, per
the requirements doc.

In [20]:
diff_frame = pd.concat([eri, X], axis=1).diff().dropna()
diff_eri = diff_frame["ERI"]
diff_X = diff_frame.drop(columns="ERI")

design_diff = add_constant(diff_X, has_constant="add")
model_diff = sm.OLS(diff_eri, design_diff).fit()

model_diff_coefs = pd.DataFrame({
    "term": model_diff.params.index,
    "coef": model_diff.params.values,
    "std_err": model_diff.bse.values,
    "t_stat": model_diff.tvalues.values,
    "p_value": model_diff.pvalues.values,
})
model_diff_coefs["significance"] = model_diff_coefs["p_value"].map(stars)
model_diff_coefs.to_csv(DIFF_OLS_COEF_OUT, index=False)
print(f"Written -> {DIFF_OLS_COEF_OUT}")
model_diff_coefs

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_b_first_differenced_ols_coefficients.csv


,term,coef,std_err,t_stat,p_value,significance
0,const,0.030579,0.029903,1.022631,0.315558,
1,DIVP,2.352928,1.379395,1.705768,0.099531,*
2,DIVM,-1.402368,1.043964,-1.343310,0.190354,
3,INF,0.000152,0.003758,0.040399,0.968073,
4,EXR,-0.003543,0.001800,-1.968141,0.059399,*
5,log(FDI),0.117277,0.053479,2.192957,0.037104,**
6,SHOCK,-0.010492,0.086296,-0.121586,0.904128,


In [21]:
model_diff_fit_stats = pd.DataFrame([{
    "package_function": "statsmodels.api.OLS",
    "n_obs": int(model_diff.nobs),
    "r_squared": model_diff.rsquared,
    "adj_r_squared": model_diff.rsquared_adj,
    "f_statistic": model_diff.fvalue,
    "f_pvalue": model_diff.f_pvalue,
    "aic": model_diff.aic,
    "bic": model_diff.bic,
}])
model_diff_fit_stats.to_csv(DIFF_OLS_FIT_OUT, index=False)
print(f"Written -> {DIFF_OLS_FIT_OUT}")
model_diff_fit_stats.T

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_b_first_differenced_ols_fit_stats.csv


,0
package_function,statsmodels.api.OLS
n_obs,34
r_squared,0.423111
adj_r_squared,0.294914
f_statistic,3.300464
f_pvalue,0.014427
aic,-30.219388
bic,-19.534864


## Side-by-side comparison table

Model A (static OLS on levels), the first-differenced OLS, and **both** ARDL long-run results --
the AIC-selected grid-search (1,2) spec (Step B2-B4) and the capped no-search (1,1) spec
(Step B8) -- assembled by regressor. Both ARDL columns are kept side by side rather than the
capped result overwriting the grid-search one, so the write-up can show the robustness check was
actually performed. Both are explicitly marked as not-bounds-test-confirmed per their respective
bounds-test results above.

In [22]:
def fmt(coef, sig):
    return f"{coef:.4f}{sig}"

a_idx = model_a_coefs.set_index("term")
d_idx = model_diff_coefs.set_index("term")
lr_idx = long_run.set_index("term")
lr_capped_idx = long_run_capped.set_index("term")

comparison_rows = []
for label in ["const"] + list(REGRESSORS.keys()):
    a_term = "const" if label == "const" else label
    d_term = "const" if label == "const" else label
    row = {"regressor": label}
    if a_term in a_idx.index:
        row["model_a_static_ols"] = fmt(a_idx.loc[a_term, "coef"], a_idx.loc[a_term, "significance"])
    if d_term in d_idx.index:
        row["first_differenced_ols"] = fmt(d_idx.loc[d_term, "coef"], d_idx.loc[d_term, "significance"])
    if label != "const" and label in lr_idx.index:
        row["ardl_long_run_grid_(1,2)"] = fmt(lr_idx.loc[label, "long_run_coef"], lr_idx.loc[label, "significance"])
    if label != "const" and label in lr_capped_idx.index:
        row["ardl_long_run_capped_(1,1)"] = fmt(lr_capped_idx.loc[label, "long_run_coef"], lr_capped_idx.loc[label, "significance"])
    comparison_rows.append(row)

comparison_table = pd.DataFrame(comparison_rows).set_index("regressor")
comparison_table.loc["theta1 (ERI.L1, ECT)", "ardl_long_run_grid_(1,2)"] = fmt(ect_coef, stars(ect_p)) + " (ECM, not a long-run coef)"
comparison_table.loc["theta1 (ERI.L1, ECT)", "ardl_long_run_capped_(1,1)"] = fmt(ect_capped_coef, stars(ect_capped_p)) + " (ECM, not a long-run coef)"
comparison_table.loc["n_obs"] = [int(model_a.nobs), int(model_diff.nobs), int(uecm.nobs), int(uecm_capped.nobs)]
comparison_table.loc["df_resid"] = ["", "", int(uecm.df_resid), int(uecm_capped.df_resid)]
comparison_table.loc["ardl_(p,q)"] = ["", "", f"({chosen_p},{chosen_q}) [AIC grid search]", f"({CAPPED_P},{CAPPED_Q}) [capped, no search]"]
comparison_table.loc["bounds_test_cointegration"] = ["", "", cointegration_status, capped_cointegration_status]
comparison_table.loc["carried_forward_to_step_vii"] = ["", "", "no", "yes"]

comparison_table.to_csv(COMPARISON_OUT)
print(f"Written -> {COMPARISON_OUT}")
comparison_table

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_comparison_side_by_side.csv


,model_a_static_ols,first_differenced_ols,"ardl_long_run_grid_(1,2)","ardl_long_run_capped_(1,1)"
regressor,,,,
const,-2.3345***,0.0306,NaN,NaN
DIVP,1.2680**,2.3529*,0.0194,-1.0451
DIVM,-0.8055,-1.4024,1.2413,1.2054
INF,-0.0041,0.0002,-0.0074,0.0022
EXR,-0.0009**,-0.0035*,0.0003,0.0001
log(FDI),0.1428***,0.1173**,-0.1869**,-0.1301**
SHOCK,-0.0203,-0.0105,-0.0330,0.0228
"theta1 (ERI.L1, ECT)",NaN,NaN,"-1.0060*** (ECM, not a long-run coef)","-0.7585*** (ECM, not a long-run coef)"
n_obs,35,34,34,34


## Decisions & flags (explicit recap)

- **ARDL max lag capped at 1-2** given the small sample. Grid restricted to `p in {1,2}`,
  `q in {1,2}` (uniform `q` across all six regressors, not independently per regressor, to keep
  the search space tractable at n≈34; `q=0` excluded because `UECM` requires `order >= 1` for
  every exogenous regressor).
- **AIC and BIC disagreed** on the optimal lag: AIC selected `(p,q)` with the lowest AIC
  (reported above), BIC selected a different, more parsimonious `(p,q)`. Per the plan's stated
  default, **AIC's choice is used** as the primary model; the BIC alternative is reported
  (Step B1) and re-estimated as a robustness check (Step B7) rather than silently discarded.
- **Degrees-of-freedom flag:** the AIC-preferred specification uses a large parameter count
  relative to the ~33-34 usable observations (reported exactly in Step B1's table) -- stated
  explicitly rather than treated as an unremarkable AIC win.
- **Bounds-test case: Case 3** (unrestricted intercept, no trend) -- stated explicitly since this
  is a common EViews-vs-Python mismatch source; matches the model's `trend="c"` specification.
- **Bounds-test result is inconclusive at 5%** (falls between the lower and upper bound) and
  leans toward no-cointegration at 1% (below the lower bound). Per the plan, this is reported as
  such and **not forced** toward either "cointegration confirmed" or "no cointegration."
  Long-run coefficients are still reported (Step B4) but explicitly caveated as not confirmed.
- **Internal-tension flag:** the error-correction term (theta1) is negative and statistically
  significant even though the bounds test does not confirm cointegration at 5%. This tension is
  flagged rather than treated as full corroboration of cointegration on its own.
- **Capped ARDL(1,1) robustness check (Step B8), run at fixed lag length with no grid search**,
  specifically to get a healthier degrees-of-freedom base than the AIC-preferred (1,2) spec's
  13-14 residual df. Result: the bounds test remains inconclusive at this leanest possible spec
  too -- reported as a genuine finding about the sample's power to resolve cointegration among
  6 regressors with ~34 annual observations, not as a dead end or a reason to keep searching for
  a spec that resolves it. The error-correction term at (1,1) is negative, significant, **and**
  within the expected (-1,0) range (an improvement on the (1,2) spec's theta1, which fell just
  outside that range) -- consistent with, not contradicting, the (1,2)/(2,1) results.
- **Which ARDL spec is carried forward to step vii:** the capped (1,1) spec, for its healthier
  degrees of freedom (see the dedicated markdown note before the comparison table). The
  grid-searched (1,2) and BIC-alternative (2,1) results are retained in their own output files
  and in the comparison table, not overwritten, so the robustness check remains visible for the
  Chapter 4 write-up.

## Conclusion

**Model A -- static OLS on levels (n=35).** `statsmodels.api.OLS`. See `model_a_static_ols_coefficients.csv` /
`model_a_static_ols_fit_stats.csv` for the exact numbers (regenerated on every run of this
notebook; do not hardcode them here). Reported as the literal-thesis baseline regardless of
branch, per the plan -- carries spurious-regression risk given the I(1) regressors
(`DIVP, DIVM, EXR, log(FDI)`, per step v), which is exactly why it is not the primary model here.

**Model B -- ARDL(p,q) bounds testing / UECM, primary model per Branch B.**
- Lag selection: AIC and BIC disagreed (see Decisions & flags above); AIC's pick, (1,2), is used
  as the grid-search default per the plan, with the BIC alternative (2,1) reported and
  cross-checked as a robustness check (Step B7) -- both reach the same substantive conclusion.
- Bounds F-test (Case 3, k=6) on the AIC-selected (1,2) spec: **inconclusive at 5%**, leaning
  toward no-cointegration at 1%. Reported explicitly rather than forced toward a conclusion.
- **Capped ARDL(1,1) robustness check (Step B8), fixed lag length, no grid search.** Run
  specifically because the AIC-preferred (1,2) spec's 13-14 residual df was too thin to trust
  the bounds-test result on its own. At (1,1) -- the leanest ARDL/UECM form possible, ~20
  residual df -- the bounds test is **still inconclusive** at 5% and 10%, and still below the
  lower bound at 1%. This is reported as a genuine finding: with ~34 annual observations and 6
  regressors, the bounds test likely lacks the power to confirm or rule out cointegration here,
  independent of lag length -- not as a dead end, and not as grounds to keep searching for a
  spec that happens to resolve it.
- Long-run coefficients: reported for both the grid-search (1,2) spec
  (`ardl_long_run_coefficients.csv`) and the capped (1,1) spec
  (`ardl_capped_1_1_long_run_coefficients.csv`), explicitly **not** treated as confirmed at
  either spec, since neither bounds test confirmed cointegration.
- Error-correction term (theta1): negative and statistically significant at both specs
  (`ardl_error_correction_term.csv`, `ardl_capped_1_1_error_correction_term.csv`) -- at the
  capped (1,1) spec it also falls within the expected (-1,0) range (an improvement on the
  (1,2) spec, where it fell just outside). Well-behaved on its own terms, but this alone does
  not override the inconclusive bounds test (flagged as an internal tension above).
- Short-run coefficients: reported as supplementary evidence for both specs
  (`ardl_short_run_coefficients.csv`, `ardl_capped_1_1_short_run_coefficients.csv`).
- **Carried forward to step vii: the capped (1,1) spec** -- healthier degrees of freedom for
  diagnostic tests, same substantive conclusion as the grid-search result, reached with less
  machinery. The (1,2)/(2,1) results are retained, not discarded (see the dedicated note above
  the comparison table).

**First-differenced OLS (comparison).** Reported alongside Model A and Model B per the
requirements doc; see `model_b_first_differenced_ols_coefficients.csv` /
`..._fit_stats.csv`.

**Side-by-side comparison:** `model_comparison_side_by_side.csv` assembles Model A, the
differenced OLS, and both ARDL specs (grid-search and capped) by regressor for the Chapter 4
draft.

### Definition of done

- [x] Model A estimated and fully reported regardless of branch.
- [x] Branch B triggered -> ARDL/UECM estimated with lag selection, bounds test, long-run
      coefficients, error-correction term, and short-run coefficients all reported, plus the
      first-differenced OLS comparison.
- [x] Every judgment call flagged in the plan is stated explicitly above, not silently resolved:
      the AIC/BIC lag tiebreak, the degrees-of-freedom concern at the AIC-preferred lag, the
      bounds-test case (3), the inconclusive bounds-test result, and the ECT-vs-bounds-test
      internal tension.
- [x] Capped ARDL(1,1) degrees-of-freedom robustness check performed (Step B8): reports df
      explicitly alongside the (1,2)/(2,1) specs, re-runs the bounds test at 1%/5%/10%, reports
      long-run coefficients and the ECT regardless of outcome, and states the still-inconclusive
      result as a small-sample power finding rather than forcing a conclusion. The grid-search
      and BIC-alternative results are kept, not overwritten, and the notebook states explicitly
      which spec (capped (1,1)) is carried forward to step vii and why.